In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv


load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is set.")
else:
    raise ValueError("OPENAI_API_KEY is not set. Please set it in the .env file or as an environment variable.")


from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

### CHAIN WITH Parallel Chains

In [ ]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")])

In [ ]:
# TASK - 2 [LLM]

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

In [ ]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [ ]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)-> dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

#### Parallel Chain 1

In [ ]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

# TASK - 2 [LLM]

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser

#### Parallel Chain 2

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

In [ ]:
def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])
    
    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

### Final Orchestration

In [ ]:
final_chain = ( 
                    prompt_template | 
                    llm_openai | 
                    str_parser | 
                    dictionary_maker_runnable |
                    RunnableParallel(branches = {"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)

In [ ]:
final_chain.invoke("KGF")

#### Chain as a Runnable

In [ ]:
# TASK - 1 [Beautify Function]

def beautify(final_response:dict)-> dict:

    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)


# TASK - 2 [Final Chain]

# final_chain 


# Beautified Chain
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")